## Getting Started with Claude Managed Agents

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Create an Agent

In [ ]:
agent = client.beta.agents.create(
    name="Demo-Managed-Agent",
    model=claude_model_name,
    system="You are a helpful AI Assistant.",
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

### Create an Environment

In [ ]:
environment = client.beta.environments.create(
    name="quickstart-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

### Start a Session

In [ ]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Quickstart session",
)

print(f"Session ID: {session.id}")

### Start an Agent Loop

In [ ]:
chat = True

while chat:
    user_query = input("Enter ""exit"" or your user query to continue")

    if user_query == "exit":
        chat = False
    else:
        with client.beta.sessions.events.stream(session.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session.id,
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": user_query,
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break